# Build Forcing Validation - Temperature
- Tony Grove

In [1]:
%matplotlib ipympl
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from gdriver.gdriver import get_credentials, getCSVtoPandas
import os

In [2]:
# Run parameters
site = 'Tony Grove RS'
latitude = 41.89
longitude = -111.57
start_date = '20150101'
end_date = '20150201'
buildforcing_version = 'v0.1'
environment = '/home/chriscox/Data/'

### Load datasets

In [3]:
# Load the forcing dataset
tg_forcing = xr.open_dataset(os.path.join(environment,'forcing',f'tonygrove_{start_date}_{end_date}_{buildforcing_version}.nc'))

In [4]:
tg_forcing

<xarray.Dataset> Size: 55kB
Dimensions:           (latitude: 1, longitude: 1, time: 768)
Coordinates:
  * latitude          (latitude) float64 8B 41.89
  * longitude         (longitude) float64 8B -111.6
  * time              (time) datetime64[ns] 6kB 2015-01-01 ... 2015-02-01T23:...
Data variables:
    LWdown            (time, latitude, longitude) float64 6kB ...
    SWdown            (time, latitude, longitude) float64 6kB ...
    Psurf             (time, latitude, longitude) float64 6kB ...
    Qair              (time, latitude, longitude) float64 6kB ...
    Tair              (time, latitude, longitude) float64 6kB ...
    Rainf             (time, latitude, longitude) float64 6kB ...
    Snowf             (time, latitude, longitude) float64 6kB ...
    Wind              (time, latitude, longitude) float64 6kB ...
    reference_height  (latitude, longitude) int64 8B ...
    canopy_height     (latitude, longitude) float64 8B ...

In [5]:
# Extract Tair from dataset to a pandas dataframe ignoring the lat/lon dimensions
# and keeping the time dimension
forcing_Tair = tg_forcing['Tair'].isel(latitude=0, longitude=0)
forcing_Tair = forcing_Tair.to_dataframe()

In [6]:
# Convert temperature from Kelvin to Celsius
forcing_Tair['Tair_C'] = forcing_Tair['Tair'] - 273.15

### Harmonize timezones
I seem to remember finding a problem with the UTC times in the gamut data. Change forcing to America/Denver to compare.


In [7]:
forcing_Tair.index = forcing_Tair.index.tz_localize('UTC').tz_convert('America/Denver')
#forcing_SWdown.index = forcing_SWdown.index.tz_localize('UTC').tz_convert('America/Denver')

### Load GAMUT

In [8]:
# Load gdriver credentials
credentials = get_credentials()

# Get the ST110 data
gamut_st110 = getCSVtoPandas('TonyGrove/LR_TG_C_AirTemp_ST110_Avg_SourceID_1_QC_1.csv',credentials,skiprows=56)

# Change index to DateTimeUTC
gamut_st110['DateTime'] = pd.to_datetime(gamut_st110['LocalDateTime'])
gamut_st110 = gamut_st110.set_index('DateTime')

# Convert -9999 to NaN
gamut_st110 = gamut_st110.replace(-9999, np.nan)

RefreshError: ('invalid_scope: Bad Request', {'error': 'invalid_scope', 'error_description': 'Bad Request'})

In [ ]:
# Extract a subset of the data for the time period of interest
start_date = '2015-01-01 00:00:00'
end_date = '2015-02-01 00:00:00'
gamut_st110_month = gamut_st110.loc[start_date:end_date]

# Localize datetime to america/denver timezone
gamut_st110_month.index = gamut_st110_month.index.tz_localize('America/Denver')

In [ ]:
# Get SW gamut data: SP230
gamut_sp230 = getCSVtoPandas('TonyGrove/LR_TG_C_SWIn_SP230_Avg_SourceID_1_QC_1.csv',credentials,skiprows=55)
gamut_sp230['DateTime'] = pd.to_datetime(gamut_sp230['LocalDateTime'])
gamut_sp230 = gamut_sp230.set_index('DateTime')
gamut_sp230 = gamut_sp230.replace(-9999, np.nan)

In [ ]:
# Extract one month of sp230 data
gamut_sp230_month = gamut_sp230.loc[start_date:end_date]
# Localize datetime to america/denver timezone
gamut_sp230_month.index = gamut_sp230_month.index.tz_localize('America/Denver')

## Comparisons

In [ ]:
# Plot Tair and compare to ST110 data
fig, ax = plt.subplots(figsize=(12, 6))
forcing_Tair['Tair_C'].plot(ax=ax, label='Forcing Tair (C)', color='blue')
gamut_st110_month['AirTemp_ST110_Avg'].plot(ax=ax, label='GAMUT Tair (C)', color='red')

#

In [ ]:
# Plot SWdown and compare to SP230 data
fig, ax = plt.subplots(figsize=(12, 6))
forcing_SWdown['SWdown'].plot(ax=ax, label='Forcing SWdown (W/m2)', color='blue')
gamut_sp230_month['SWIn_SP230_Avg'].plot(ax=ax, label='GAMUT SWdown (W/m2)', color='red')